# Final Model Selection

**Set `CHOICE`** to the model key you picked after the final evaluation (`reports/final_evaluation_report.md`), e.g. `'catboost_model'`, `'random_forest'`, `'xgboost_model'`, `'lightgbm_model'`. The model is built with its final configuration from `reports/final_configs.json` (tuned params or library defaults), evaluated once on validation + test, and saved to `models/final_model.pkl` + `models/preprocessor.pkl`.

Without `reports/final_configs.json`, the notebook falls back to the lowest repeated-CV tuned model.

In [ ]:
CHOICE = None   # <- set to a model key, e.g. 'catboost_model'

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings('ignore')
import pandas as pd
from src.preprocessing import prepare_data
from src.models import make_model, display_name
from src.tuning import build_tuned_model, load_tuned_params, build_final_model, final_params, FINAL_CONFIG_FILE
from src.evaluation import evaluate, results_table, print_summary
from src.prediction import save_final
from src import config

In [ ]:
data = prepare_data()
n_rows = sum(data['meta']['rows'][s] for s in ('train','val','test'))

In [ ]:
if FINAL_CONFIG_FILE.exists():
    evidence = config.REPORTS/'final'/'final_repeated_cv.csv'
    if evidence.exists():
        display(pd.read_csv(evidence)[['key','model','rep_cv_rmse_5x3','val_rmse','val_mae','test_rmse','test_mae','test_r2']])
    if CHOICE is None:
        raise ValueError('Set CHOICE to the model key you decided on (see the table above / reports/final_evaluation_report.md).')
    model, cfg = build_final_model(CHOICE, data['target_transform'])
    best_key, rule = CHOICE, 'chosen after final evaluation (reports/final_configs.json)'
    params = final_params(CHOICE)
    label = f'{display_name(CHOICE)} ({cfg})'
else:
    rep = pd.read_csv(config.REPORTS/'tuning'/'repeated_cv_top_tuned.csv').sort_values('rep_cv_rmse_5x3')
    best_key, rule = rep.iloc[0]['key'], 'lowest repeated 5x3 CV RMSE'
    base = best_key.removesuffix('_tuned'); params = load_tuned_params().get(base, {})
    model = build_tuned_model(base, data['target_transform'], params); label = f'{display_name(base)} (tuned)'
print('Selected:', label, '|', rule); print('Hyperparameters:', params or 'library defaults')

In [ ]:
results, pred, _ = evaluate(model, data, label, best_key, evaluate_test=True)
print_summary(results, n_rows, len(data['feature_names']))
results_table(results)

In [ ]:
meta = {'model': label, 'key': best_key, 'selection_rule': rule, 'hyperparameters': params,
        'reference_id': data['fingerprint']['reference']['reference_id'],
        'cv_rmse': results['cv']['RMSE_mean'], 'val_rmse': results['val']['RMSE'], 'val_mae': results['val']['MAE'],
        'test_rmse': results['test']['RMSE'], 'test_mae': results['test']['MAE'],
        'test_r2': results['test']['R2'], 'test_mape': results['test']['MAPE'],
        'target_transform': data['target_transform']}
save_final(model, data['preprocessor'], data['feature_names'], meta)
print('Saved models/final_model.pkl + models/preprocessor.pkl'); meta